# Assignment 2: Character-Level Bigram Language Model

This notebook trains a small next-character model on an included name-like text corpus. It builds a character vocabulary, encodes each transition as a character pair, uses a reproducible 80/20 train-validation split, records mean negative log-likelihood before and after fitting, and compares generated samples.

The implementation uses only the Python standard library. Run cells from top to bottom. The saved model and results are written alongside this notebook.

In [ ]:
from pathlib import Path
import json
import runpy

PROJECT = Path.cwd()
if not (PROJECT / 'train.py').exists():
    PROJECT = Path('Assignment 2 - Language Model Training')
assert (PROJECT / 'train.py').exists(), 'Open this notebook from its project folder.'
print((PROJECT / 'data' / 'names.txt').read_text(encoding='utf-8'))

## Character vocabulary and encoding

Each distinct character is assigned a stable integer ID. Newline marks the end of an example, while a special start token begins each name.

In [ ]:
names = [s.strip().lower() for s in (PROJECT / 'data' / 'names.txt').read_text(encoding='utf-8').splitlines() if s.strip()]
vocabulary = sorted(set(''.join(names)) | {'\n'})
stoi = {ch: i for i, ch in enumerate(vocabulary)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda text: [stoi[ch] for ch in text]
decode = lambda ids: ''.join(itos[i] for i in ids)
example = names[0]
assert decode(encode(example)) == example
print('Vocabulary:', ''.join('\\n' if c == '\n' else c for c in vocabulary))
print('Round trip:', example, '->', encode(example), '->', decode(encode(example)))

## Fit the model and evaluate held-out examples

The trainer counts adjacent character pairs in the training partition only. Add-one smoothing makes unseen transitions possible. Loss is mean negative log-likelihood per next-character prediction; lower is better. The uniform baseline is evaluated before any transitions are learned.

In [ ]:
runpy.run_path(str(PROJECT / 'train.py'), run_name='__main__')
losses = json.loads((PROJECT / 'results' / 'losses.json').read_text(encoding='utf-8'))
print(json.dumps(losses, indent=2))

## Compare generations and inspect the saved model

The random baseline samples characters uniformly. The trained generator uses the learned one-character transition probabilities. These are pattern samples, not meaningful language; with a small corpus and a one-character context, some outputs will be imperfect.

In [ ]:
print((PROJECT / 'results' / 'samples.txt').read_text(encoding='utf-8'))
model = json.loads((PROJECT / 'model' / 'bigram_model.json').read_text(encoding='utf-8'))
print('Saved model:', model['kind'], '| vocabulary size:', len(model['vocabulary']), '| smoothing:', model['smoothing'])